# Ollama + RAG: Chat with your PDF

This notebook is designed for **Google Colab**. It installs Ollama in the Colab runtime, downloads a small language model and embedding model, lets you upload a PDF, indexes it in ChromaDB, and answers questions using retrieved passages.

> Keep this Colab session open while using the chatbot. Model downloads may take several minutes.

## 1. Install the required packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y curl zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install ollama chromadb pypdf

## 2. Start Ollama and download models

We use `qwen2.5:3b` for answers and `nomic-embed-text` for embeddings.

In [15]:
import subprocess, time, os, requests

# Start the Ollama server in the background using absolute path
ollama_server = subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait until the local API is ready
for attempt in range(30):
    try:
        requests.get('http://127.0.0.1:11434/api/tags', timeout=2)
        break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start. Restart the runtime and run this cell again.')

print('Ollama is running!')
!/usr/local/bin/ollama pull qwen2.5:3b
!/usr/local/bin/ollama pull nomic-embed-text

Ollama is running!




## 3. Test the language model

In [7]:
import ollama

response = ollama.generate(
    model='qwen2.5:3b',
    prompt='Explain Retrieval-Augmented Generation (RAG) in three simple sentences.'
)
print(response['response'])

Retrieval-Augmented Generation (RAG) is a method that combines the strengths of traditional generation models with the capability to retrieve and utilize relevant information from a database or knowledge base to enhance the quality and relevance of the generated content.


## 4. Upload a PDF

Click **Choose Files** and select a PDF from your computer.

In [10]:
# Since you don't have a PDF, let's download a sample PDF to test the chatbot
import urllib.request

sample_pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"  # The famous 'Attention Is All You Need' paper
save_path = "attention_paper.pdf"

print("Downloading sample PDF...")
urllib.request.urlretrieve(sample_pdf_url, save_path)
print(f"Sample PDF downloaded and saved as '{save_path}'!")

Sample PDF downloaded and saved as 'attention_paper.pdf'!


In [11]:
# Automatically load the downloaded sample PDF
from pypdf import PdfReader

PDF_PATH = "attention_paper.pdf"
reader = PdfReader(PDF_PATH)
pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ''
    if text.strip():
        pages.append({'page': page_number, 'text': text})

print(f'Read {len(pages)} pages containing text from {PDF_PATH}')
print(pages[0]['text'][:1000] if pages else 'No extractable text found.')

Read 15 pages containing text from attention_paper.pdf
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, di

In [8]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
pdf_files = [name for name in uploaded if name.lower().endswith('.pdf')]
if not pdf_files:
    raise ValueError('Please upload at least one PDF file.')

PDF_PATH = pdf_files[0]
reader = PdfReader(PDF_PATH)
pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ''
    if text.strip():
        pages.append({'page': page_number, 'text': text})

print(f'Read {len(pages)} pages containing text from {PDF_PATH}')
print(pages[0]['text'][:1000] if pages else 'No extractable text found. This may be a scanned PDF requiring OCR.')

KeyboardInterrupt: 

## 5. Split the document into chunks

Chunks are small text sections. Overlap helps preserve context between neighboring chunks.

In [12]:
def chunk_text(text, chunk_size=900, overlap=150):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        piece = ' '.join(words[start:end]).strip()
        if piece:
            chunks.append(piece)
        if end == len(words):
            break
        start = end - overlap
    return chunks

chunks = []
metadatas = []
for item in pages:
    page_chunks = chunk_text(item['text'])
    for chunk_index, chunk in enumerate(page_chunks):
        chunks.append(chunk)
        metadatas.append({'source': PDF_PATH, 'page': item['page'], 'chunk': chunk_index})

print(f'Created {len(chunks)} chunks')
print(chunks[0][:500] if chunks else 'No chunks created')

Created 15 chunks
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu Łukasz 


## 6. Generate embeddings and store them in ChromaDB

In [16]:
import chromadb

client = chromadb.Client()
# Create a fresh collection for this notebook run
try:
    client.delete_collection('pdf_rag')
except Exception:
    pass
collection = client.create_collection(name='pdf_rag')

def get_embedding(text):
    result = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return result['embedding']

# Embed and store in small batches
batch_size = 32
for start in range(0, len(chunks), batch_size):
    batch_chunks = chunks[start:start + batch_size]
    batch_meta = metadatas[start:start + batch_size]
    batch_embeddings = [get_embedding(chunk) for chunk in batch_chunks]
    ids = [f'chunk-{i}' for i in range(start, start + len(batch_chunks))]
    collection.add(
        ids=ids,
        documents=batch_chunks,
        metadatas=batch_meta,
        embeddings=batch_embeddings
    )
    print(f'Indexed {min(start + batch_size, len(chunks))}/{len(chunks)} chunks')

print('PDF indexing complete!')

Indexed 15/15 chunks
PDF indexing complete!


## 7. Retrieve relevant passages

This function embeds your question and searches ChromaDB for the closest document chunks.

In [17]:
def retrieve(question, top_k=4):
    question_embedding = get_embedding(question)
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=min(top_k, collection.count()),
        include=['documents', 'metadatas', 'distances']
    )
    documents = results['documents'][0] if results['documents'] else []
    metadata = results['metadatas'][0] if results['metadatas'] else []
    return list(zip(documents, metadata))

question = 'What is the main topic of this document?'
for passage, meta in retrieve(question):
    print(f"Page {meta['page']}: {passage[:350]}...\n")

Page 14: Input-Input Layer5 The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . <EOS> <pad> The Law will never be perfect , but its application should be just - this is what we are missing , in my opinion . <EOS> <pad> Input-Input Layer5 The Law will never be perfect , but its application should...

Page 13: Attention Visualizations Input-Input Layer5 It is in this spirit that a majority of American governments have passed new laws since 2009 making the registration or voting process more difficult . <EOS> <pad> <pad> <pad> <pad> <pad> <pad> It is in this spirit that a majority of American governments have passed new laws since 2009 making the registra...

Page 9: Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-German translation development set, newstest2013. Listed perplexities are per-wordpiece, according to our byte-pair enc

## 8. Ask Ollama using the retrieved context

The model is instructed to answer from the PDF context and say when the answer is not present.

In [18]:
def ask_pdf(question, top_k=4):
    matches = retrieve(question, top_k=top_k)
    if not matches:
        return 'The document has no indexed text to search.'

    context_parts = []
    for passage, meta in matches:
        context_parts.append(f"[Page {meta['page']}]\n{passage}")
    context = '\n\n'.join(context_parts)

    prompt = f"""You are a helpful assistant answering questions about an uploaded PDF.
Use ONLY the context below. If the answer is not in the context, say you could not find it in the PDF.
When possible, mention the page number supporting your answer.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    result = ollama.generate(model='qwen2.5:3b', prompt=prompt)
    return result['response'].strip()

print(ask_pdf('What is the main topic of this document?'))

The main topic of this document is not explicitly stated. However, based on the content provided, the document appears to discuss aspects of the Transformer model, including attention mechanisms, model architectures, and performance on translation and parsing tasks. Specifically, it covers attention visualizations, architecture variations, and experiments related to the Transformer model. It also mentions experiments on English constituency parsing.


## 9. Chat with your PDF

Run this cell, enter questions, and type `exit` when you are finished.

In [19]:
while True:
    question = input('Ask a question about your PDF (or type exit): ').strip()
    if question.lower() in {'exit', 'quit'}:
        print('Goodbye!')
        break
    if question:
        print('\nAnswer:', ask_pdf(question), '\n')

Ask a question about your PDF (or type exit): what is the transformer

Answer: The Transformer is a sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. It is proposed as a new network architecture. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. The Transformer achieves state-of-the-art results on the WMT 2014 English-to-German and English-to-French translation tasks. Additionally, it successfully generalizes to other tasks like English constituency parsing, even with limited training data. 

Ask a question about your PDF (or type exit): wht is minecraft

Answer: Could not find in the PDF. 

Ask a question about your PDF (or type exit): who are the authors of this paper

Answer: The authors of this paper are Ashish Vaswani, Noam Shazeer, Niki Parm

## 10. What you built

- **Ollama** runs the language and embedding models locally inside this Colab runtime.
- **Chunking** splits the PDF text into searchable pieces.
- **Embeddings** represent each chunk and question as vectors.
- **ChromaDB** stores vectors and retrieves similar chunks.
- **RAG** sends the retrieved passages to the language model so it can answer using your document.

**Limitations:** This starter notebook extracts selectable text from PDFs. Scanned/image-only PDFs need OCR. The Colab runtime is temporary, so the downloaded models and in-memory ChromaDB collection may disappear when the runtime resets.